# History buffer

> Per-env history of the infos, for policies that give past observations to their world model.

In [ ]:
#| default_exp buffer

In [ ]:
#| hide
from nbdev.showdoc import *

`HistoryBuffer` is stable-worldmodel's, with one multi-agent change: the **block keys** (the
actions executed between two returned frames) keep the agent axis,
``(n_envs, n - 1, num_agents, action_block * D)``, where ``D`` is the size of one agent's action
(1 for a discrete action index).

In [ ]:
#| export
# Adapted from stable-worldmodel's `buffer.py` (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from collections import deque
from typing import Any, Iterable

import numpy as np
import torch

In [ ]:
#| export
class HistoryBuffer:
    """
    Per-env ring buffer over batched info dicts (values with a leading env axis, e.g. the
    ``(n_envs, 1, ...)`` infos of an :class:`EnvPool`).

    :meth:`append` stores one entry per env; :meth:`get` returns the last ``n`` entries, every
    ``action_block`` env steps, oldest first: a key with per-step shape ``(n_envs, T, ...)``
    comes out as ``(n_envs, n * T, ...)``.

    Keys in ``block_keys`` (typically ``('action',)``) come out as the ``n - 1`` blocks of
    actions executed *between* consecutive returned frames:
    ``(n_envs, n - 1, num_agents, action_block * D)``, chronological within a block, NaNs (the
    reset's action) zeroed, omitted when ``n == 1``.

    .. warning:: While an env holds fewer than ``(n - 1) * action_block + 1`` entries (just after
        a reset), its history is left-padded with copies of its oldest entry and zero action
        blocks, as if it had stood still before the episode. :meth:`num_strided` tells how much
        real history each env has.
    """
    def __init__(self, n_envs: int, max_len: int, action_block: int = 1, block_keys: Iterable[str] = ()):
        for name, value in (('n_envs', n_envs), ('max_len', max_len), ('action_block', action_block)):
            if value <= 0:
                raise ValueError(f"{name} must be positive, got {value}")
        self.n_envs, self.max_len, self.action_block = n_envs, max_len, action_block
        self.block_keys = frozenset(block_keys)
        self._buffers: list[deque[dict[str, Any]]] = [deque(maxlen=max_len) for _ in range(n_envs)]

    def append(self, info_dict: dict[str, Any]):
        "Store the entry of every env of a batched info dict."
        for i in range(self.n_envs):
            self._buffers[i].append({k: _slice_env(v, i) for k, v in info_dict.items()})

    def get(self, n: int, env_ids: list[int] | None = None) -> dict[str, Any] | None:
        "The last `n` strided entries of the envs `env_ids` (all by default), padded during warm-up."
        if n <= 0:
            return None
        buffers = self._buffers if env_ids is None else [self._buffers[i] for i in env_ids]
        if any(len(buf) == 0 for buf in buffers):
            return None
        out: dict[str, Any] = {}
        for key in buffers[0][-1]:
            if key in self.block_keys:
                if n > 1:
                    out[key] = self._get_blocked(key, n, buffers)
                continue
            per_env = []
            for buf in buffers:
                k = min(n, (len(buf) - 1) // self.action_block + 1)
                entries = [buf[-1 - off * self.action_block][key] for off in reversed(range(k))]
                per_env.append(_merge_time([entries[0]] * (n - k) + entries))
            out[key] = _stack_envs(per_env)
        return out

    def _get_blocked(self, key: str, n: int, buffers) -> Any:
        ab, per_env = self.action_block, []
        for buf in buffers:
            k = min(n, (len(buf) - 1) // ab + 1)
            blocks = []
            for i in range(k - 1):
                start = (k - 2 - i) * ab          # frame i + 1, counted from the newest entry
                blocks.append(_agent_block([buf[-1 - off][key] for off in reversed(range(start, start + ab))]))
            zero = _agent_block([buf[-1][key]] * ab) * 0
            per_env.append(_stack_envs([zero] * (n - 1 - len(blocks)) + blocks))
        return _stack_envs(per_env)

    def num_strided(self, env_ids: list[int] | None = None) -> list[int]:
        "Number of real (unpadded) strided entries held by each env."
        ids = range(self.n_envs) if env_ids is None else env_ids
        return [(len(self._buffers[i]) - 1) // self.action_block + 1 if self._buffers[i] else 0 for i in ids]

    def reset(self, env_ids: list[int] | None = None):
        "Clear the buffers of `env_ids` (all by default)."
        for i in (range(self.n_envs) if env_ids is None else env_ids):
            self._buffers[i].clear()

    def __len__(self) -> int:
        return min(len(buf) for buf in self._buffers)


def _slice_env(v: Any, i: int) -> Any:
    return v[i] if torch.is_tensor(v) or isinstance(v, np.ndarray) else v


def _agent_block(entries: list) -> Any:
    """
    One block of per-step entries ``(T, num_agents, *D)``: ``(num_agents, len(entries) * T * prod(D))``,
    chronological per agent, NaNs zeroed.
    """
    merged = _merge_time(entries)                       # (ab * T, num_agents, *D)
    if torch.is_tensor(merged):
        merged = merged.reshape(merged.shape[0], merged.shape[1], -1).transpose(0, 1).reshape(merged.shape[1], -1)
        return torch.nan_to_num(merged, nan=0.0)
    merged = np.asarray(merged)
    merged = merged.reshape(merged.shape[0], merged.shape[1], -1).swapaxes(0, 1).reshape(merged.shape[1], -1)
    return np.nan_to_num(merged, nan=0.0)


def _merge_time(xs: list) -> Any:
    first = xs[0]
    if torch.is_tensor(first):
        return torch.stack(xs) if first.ndim == 0 else torch.cat(xs)
    if isinstance(first, np.ndarray):
        return np.stack(xs) if first.ndim == 0 else np.concatenate(xs)
    return np.array(xs)


def _stack_envs(xs: list) -> Any:
    first = xs[0]
    if torch.is_tensor(first):
        return torch.stack(xs)
    return np.stack(xs) if isinstance(first, np.ndarray) else np.array(xs)

### Tests

In [ ]:
buf = HistoryBuffer(n_envs=2, max_len=10, action_block=2, block_keys=('action',))
for t in range(5):   # env steps 0..4: frames are kept every 2 steps
    buf.append({'pov': torch.full((2, 1, 3, 4, 4, 3), t),                    # (n_envs, T=1, agents, ...)
                'action': torch.tensor([[[float(t), 10. + t, 20. + t]]] * 2) if t else torch.full((2, 1, 3), float('nan'))})
h = buf.get(3)
assert h['pov'].shape == (2, 3, 3, 4, 4, 3) and h['pov'][0, :, 0, 0, 0, 0].tolist() == [0, 2, 4]
# blocks of the 2 actions executed between frames (0, 2) and (2, 4), per agent
assert h['action'].shape == (2, 2, 3, 2)
assert h['action'][0, :, 0].tolist() == [[1, 2], [3, 4]] and h['action'][0, :, 1].tolist() == [[11, 12], [13, 14]]
assert buf.num_strided() == [3, 3]

buf.reset([1]); buf.append({'pov': torch.zeros(2, 1, 3, 4, 4, 3), 'action': torch.zeros(2, 1, 3)})
h = buf.get(3, env_ids=[1])        # env 1 has one real frame: padded with copies and zero blocks
assert h['pov'].shape == (1, 3, 3, 4, 4, 3) and (h['action'] == 0).all() and buf.num_strided([1]) == [1]
assert 'action' not in buf.get(1)  # no blocks with a single frame

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()